# Анализ результатов сценариев и ИПУ

Жизненный цикл ФП/СФП:

- **Вариант 1:** Возникновение ФП/СФП → Сценарий → Результат сценария → Закрытие
- **Вариант 2:** Возникновение ФП/СФП → Сценарий → Негативный результат → ИПУ → Результат ИПУ → Закрытие

Дедупликация: если нет дат ИПУ — берём запись с самой поздней датой сценария;  
если есть даты ИПУ — берём запись с самой поздней датой ИПУ.

## 0. Конфигурация, загрузка, нормализация, классификация

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("OK")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})
    print("Переименована колонка DESC_TEXT.1 → DESC_TEXT_1")

print(f"Загружено: {len(raw):,} строк")

if "VAL" in raw.columns:
    raw = raw[raw["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(raw):,}")

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(raw):,}")

raw["segment"] = raw["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
raw = raw[raw["segment"].notna()].copy()

raw["fp_num"]  = raw["NUMBER_FP_SFP"].str.strip()
raw["fp_type"] = raw["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
print(f"TYPE_FP значения: {raw['fp_type'].value_counts().to_dict()}")

_LAT2CYR = str.maketrans({
    'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'H': 'Н', 'K': 'К',
    'M': 'М', 'O': 'О', 'P': 'Р', 'T': 'Т', 'X': 'Х',
    'a': 'а', 'c': 'с', 'e': 'е', 'o': 'о', 'p': 'р', 'x': 'х', 'y': 'у',
})

def norm_text(s):
    if pd.isna(s) or s == "":
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = re.sub(r'[\s\xa0\u200b\ufeff]+', ' ', s)
    return s.strip()

raw["scenario"] = raw["DESC_TEXT_1"].apply(norm_text)
raw["ipu"]      = raw["DESC_TEXT"].apply(norm_text)

for col in SCR_DATE_COLS + IPU_DATE_COLS + ["DATE_END_FP_SFP"]:
    raw[f"_{col}_dt"] = pd.to_datetime(raw[col], dayfirst=True, errors="coerce")

print(f"Итого: {len(raw):,} строк, {raw['inn'].nunique():,} ИНН")

In [ ]:
SCR_POSITIVE = [
    "ФП/СФП устранен", "ФП/СФП урегулирован", "Микро_У ФП урегулирован",
    "МСБ_Принято решение УОБ о снятии ФП с контроля",
    "МСБ_Принято решение УЛ о снятии ФП с контроля",
    "1_ККБ/МСБ_ФП устранен", "СФП устранен",
    "Микро_У Требуется принятие решения УЛ о снятии фактора с контроля",
    "Принято решение УЛ о снятии ФП с контроля",
    "Микро_У Фактор устранен", "Денежная санкция уплачена",
    "ФП/СФП устранен (договор закрыт)", "Условие выполнено",
    "2_ККБ/МСБ_СФП устранен",
    "ДКБ_Принято решение УЛ о снятии ФП с контроля",
    "ДКБ_Принято решение УОБ об урегулировании ФП",
    "ДКБ_Принято решение УОБ о снятии ФП с контроля",
    "ФП устранен (договор закрыт)",
    "ДКБ_Не требуется решение УЛ/УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии фактора с контроля",
    "Микро_У ФП устранен", "ФП устранен",
    "ДКБ_ФП не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, возможно снятие ФП с контроля",
    "Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП устранен",
    "ФП урегулирован, требуется принятие решения УОБ о снятии ФП с контроля",
    "ДКБ_СФП устранен",
    "Микро_У Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП урегулирован",
    "Микро_У Принято решение УЛ о снятии ФП с контроля",
]

SCR_NEGATIVE = [
    "ФП/СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "3_ККБ/МСБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "ДКБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "12_ККБ/МСБ_ФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ДКБ_ФП оказывает негативное влияние, требует рассмотрения УОБ вопроса о признании задолженности проблемной",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
    "ФП не устранен. Лимит снижен до \"0\". Лимит восстановлению не подлежит.",
    "Микро_У Техническое закрытие ФП в связи с запуском нового сценария по СФП (СФП 15.2У/15.2.1У)",
]

SCR_ERROR = [
    "0_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "0. Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "ДКБ_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "Микро_У Информация проверена, отсутствуют основания для выявления ФП/СФП",
]

SCR_NEUTRAL = [
    "ФП не устранен, не оказывает негативного влияния, требует рассмотрения вопроса на УОБ с целью снятия ФП с контроля",
    "Активный (срок окончания, предусмотренный Порядком не наступил, находится в стадии реализации)",
    "11_ККБ/МСБ_ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "5_ МСБ_Право Банка не реализовано, ФП снят с контроля",
    "6_ККБ/МСБ_Техническое закрытие ФП",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Активный (срок окончания, предусмотренный Порядком наступил, результат реализации по состоянию на дату Реестра отсутствует)",
    "Микро_У ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "8_ККБ/МСБ_В отношении ФП принято решение УОБ",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Установлена коммерческая ставка",
    "Микро_У Фактор устранен - решение УОБ о снятии фактора с контроля",
    "9_ККБ/МСБ_В отношении ФП принято решение УЛ",
    "ДКБ_Информация проверена, выявлен соответствующий ФП/СФП",
    "Микро_У Принятие решения УЛ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ/УЛ о снятии ФП с контроля",
    "По ФП реализовано право Банка",
]

IPU_POSITIVE = [
    "ФП устранен",
    "ФП устранен (договор закрыт)",
    "Принято решение УОБ о снятии с контроля",
]
IPU_NEGATIVE = [
    "ФП не устранен, выявлены предпосылки, требующие рассмотрения вопроса о признании задолженности проблемной",
]
IPU_ACTIVE = [
    "Активный (продлен первоначальный срок окончания Плана, находится в стадии реализации)",
    "Активный (первоначально установленный срок окончания плана не наступил, находится в стадии реализации)",
    "ФП не устранен в рамках Плана, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком (требует корректировки ИПУ)",
]
IPU_ERROR = [
    "Активный (срок окончания Плана истек, результат реализации по состоянию на дату Реестра отсутствует)",
]
IPU_NEUTRAL = []

def norm_list(lst):
    return [norm_text(v) for v in lst]

SCR_POSITIVE = norm_list(SCR_POSITIVE)
SCR_NEGATIVE = norm_list(SCR_NEGATIVE)
SCR_NEUTRAL  = norm_list(SCR_NEUTRAL)
SCR_ERROR    = norm_list(SCR_ERROR)
IPU_POSITIVE = norm_list(IPU_POSITIVE)
IPU_NEGATIVE = norm_list(IPU_NEGATIVE)
IPU_ACTIVE   = norm_list(IPU_ACTIVE)
IPU_ERROR    = norm_list(IPU_ERROR)
IPU_NEUTRAL  = norm_list(IPU_NEUTRAL)

_scr_sets = {
    "положительный": set(SCR_POSITIVE),
    "отрицательный": set(SCR_NEGATIVE),
    "нейтральный":   set(SCR_NEUTRAL),
    "ошибка":        set(SCR_ERROR),
}

_ipu_sets = {
    "положительный": set(IPU_POSITIVE),
    "отрицательный": set(IPU_NEGATIVE),
    "активный":      set(IPU_ACTIVE),
    "ошибка":        set(IPU_ERROR),
    "нейтральный":   set(IPU_NEUTRAL),
}


def classify_scr(val):
    for cls, s in _scr_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"


def classify_ipu(val):
    for cls, s in _ipu_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"


# Диагностика
all_scr_classified = set(SCR_POSITIVE) | set(SCR_NEGATIVE) | set(SCR_NEUTRAL) | set(SCR_ERROR) | {""}
all_ipu_classified = set(IPU_POSITIVE) | set(IPU_NEGATIVE) | set(IPU_ACTIVE) | set(IPU_ERROR) | set(IPU_NEUTRAL) | {""}

scr_unknown = set(raw["scenario"].unique()) - all_scr_classified
ipu_unknown = set(raw["ipu"].unique()) - all_ipu_classified

if scr_unknown:
    print(f"⚠ Неклассифицированные результаты СЦЕНАРИЯ ({len(scr_unknown)}):")
    for v in sorted(scr_unknown):
        print(f"  {(raw['scenario'] == v).sum():>5,}  «{v}»  repr={repr(v)}")
else:
    print("✓ Все результаты сценария классифицированы")

if ipu_unknown:
    print(f"⚠ Неклассифицированные результаты ИПУ ({len(ipu_unknown)}):")
    for v in sorted(ipu_unknown):
        print(f"  {(raw['ipu'] == v).sum():>5,}  «{v}»  repr={repr(v)}")
else:
    print("✓ Все результаты ИПУ классифицированы")

print(f"\nСценарий: {len(SCR_POSITIVE)} полож., {len(SCR_NEGATIVE)} отриц., "
      f"{len(SCR_NEUTRAL)} нейтр., {len(SCR_ERROR)} ошибок")
print(f"ИПУ: {len(IPU_POSITIVE)} полож., {len(IPU_NEGATIVE)} отриц., "
      f"{len(IPU_ACTIVE)} актив., {len(IPU_ERROR)} ошибок")

## 1. Финализированная дедупликация (два потока)

- **Поток 1 (только сценарий):** нет ни одной заполненной даты ИПУ → берём запись с `max(END_DATE_SCR_FCT, END_DATE_SCR_PLAN)`
- **Поток 2 (сценарий + ИПУ):** есть хотя бы одна дата ИПУ → берём запись с `max(FIRST_END_DATE_EVENT, NEW_PLAN_END_DATE_EVT, END_EVENT_DATE_FACT)`
- Карточки без каких-либо дат — исключаем

In [ ]:
scr_dt = [f"_{c}_dt" for c in SCR_DATE_COLS]
ipu_dt = [f"_{c}_dt" for c in IPU_DATE_COLS]

raw["_scr_max"] = raw[scr_dt].max(axis=1)
raw["_ipu_max"] = raw[ipu_dt].max(axis=1)

# Для каждой группы: есть ли хоть одна заполненная дата ИПУ
grp_has_ipu = raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()

stream1_raw = raw[~grp_has_ipu].copy()
stream2_raw = raw[grp_has_ipu].copy()

# Поток 1: сортировка по самой поздней дате сценария
stream1_raw["_sort_dt"] = stream1_raw["_scr_max"]
stream1_raw = stream1_raw[stream1_raw.groupby(DEDUP_KEY)["_sort_dt"].transform("max").notna()]
stream1_sorted = stream1_raw.sort_values(
    ["_sort_dt"], ascending=False, na_position="last"
)
df_stream1 = stream1_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream1["stream"] = "scenario"

# Поток 2: сортировка по самой поздней дате ИПУ
stream2_raw["_sort_dt"] = stream2_raw["_ipu_max"]
stream2_raw = stream2_raw[stream2_raw.groupby(DEDUP_KEY)["_sort_dt"].transform("max").notna()]
stream2_sorted = stream2_raw.sort_values(
    ["_sort_dt"], ascending=False, na_position="last"
)
df_stream2 = stream2_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream2["stream"] = "ipu"

df = pd.concat([df_stream1, df_stream2], ignore_index=True)

# Классификация
df["scr_class"] = df["scenario"].apply(classify_scr)
df["ipu_class"] = df["ipu"].apply(classify_ipu)

total_raw_cards = raw.drop_duplicates(DEDUP_KEY).shape[0]
n1 = len(df_stream1)
n2 = len(df_stream2)
excluded_no_dates = total_raw_cards - n1 - n2

print(f"Всего уникальных карточек ФП (до фильтрации): {total_raw_cards:,}")
print(f"Поток 1 (только сценарий):  {n1:,} ({n1/total_raw_cards*100:.1f}%)")
print(f"Поток 2 (сценарий + ИПУ):   {n2:,} ({n2/total_raw_cards*100:.1f}%)")
print(f"Исключено (нет дат):         {excluded_no_dates:,} ({excluded_no_dates/total_raw_cards*100:.1f}%)")

# Исключаем карточки с пустым результатом сценария
empty_scr = (df["scr_class"] == "[пусто]").sum()
df = df[df["scr_class"] != "[пусто]"].copy()
print(f"Исключено (пустой результат сценария): {empty_scr:,}")

# Исключаем неклассифицированные результаты
uncl_scr = (df["scr_class"] == "неклассифицированный").sum()
df = df[df["scr_class"] != "неклассифицированный"].copy()
print(f"Исключено (неклассифицированный результат): {uncl_scr:,}")

print(f"\nИтого в датасете df:        {len(df):,}")

## 2. Распределение результатов сценариев

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ СЦЕНАРИЕВ (все карточки)")
print("=" * 70)

n_total = len(df)

# Сводка по классам
scr_cls = df["scr_class"].value_counts()
scr_cls_pct = (scr_cls / n_total * 100).round(1)
scr_summary = pd.DataFrame({"Количество": scr_cls, "%": scr_cls_pct})
scr_summary.index.name = "Класс результата"
print(f"\nВсего карточек: {n_total:,}\n")
display(scr_summary)

# Отдельная таблица для каждой группы
for cls_name in ["положительный", "отрицательный", "нейтральный", "ошибка"]:
    sub = df[df["scr_class"] == cls_name]
    if len(sub) == 0:
        continue
    detail = sub["scenario"].value_counts().reset_index()
    detail.columns = ["Результат сценария", "Количество"]
    detail["% от группы"] = (detail["Количество"] / len(sub) * 100).round(1)
    detail["% от всех"] = (detail["Количество"] / n_total * 100).round(2)
    print(f"\n{'─'*60}")
    print(f"  {cls_name.upper()} ({len(sub):,} карточек, {len(sub)/n_total*100:.1f}%)")
    print(f"{'─'*60}")
    display(detail.style.hide(axis="index"))

### 2.1. Проверка: ФП с результатом «ошибка» — актуальная ли запись?

Результаты из группы «ошибка» (`0. Информация проверена, отсутствуют основания для выявления ФП/СФП` и аналогичные) означают, что ФП был выявлен ошибочно.  
Проверяем: есть ли у этих карточек ФП другие записи в `raw` с иным (не ошибочным) результатом сценария?

In [ ]:
err_cards = df[df["scr_class"] == "ошибка"][DEDUP_KEY].copy()
n_err = len(err_cards)
print(f"Карточек с результатом «ошибка»: {n_err:,}")

# Ищем все записи этих карточек в raw (до дедупликации)
err_all = raw.merge(err_cards, on=DEDUP_KEY, how="inner")
err_all["_scr_cls"] = err_all["scenario"].apply(classify_scr)

# Есть ли у них записи с другим (не ошибочным) результатом?
err_other = err_all[err_all["_scr_cls"] != "ошибка"]
cards_with_other = err_other.drop_duplicates(DEDUP_KEY).shape[0]

print(f"Записей по этим карточкам в raw: {len(err_all):,}")
print(f"Из них с НЕ ошибочным результатом: {len(err_other):,}")
print(f"Карточек, у которых есть другой результат: {cards_with_other:,} из {n_err:,} "
      f"({cards_with_other/max(n_err,1)*100:.1f}%)")

if cards_with_other > 0:
    print("\nКакие альтернативные результаты у этих карточек:")
    alt = err_other["_scr_cls"].value_counts()
    for cls, cnt in alt.items():
        print(f"  {cls}: {cnt:,}")
else:
    print("\n→ Все карточки с «ошибкой» не имеют других записей — результат актуален.")

## 3. Распределение результатов ИПУ

In [ ]:
df_ipu_all = df[df["stream"] == "ipu"].copy()
empty_ipu = (df_ipu_all["ipu_class"] == "[пусто]").sum()
uncl_ipu = (df_ipu_all["ipu_class"] == "неклассифицированный").sum()
df_ipu = df_ipu_all[~df_ipu_all["ipu_class"].isin(["[пусто]", "неклассифицированный"])].copy()
n_ipu = len(df_ipu)
print(f"Карточек на этапе ИПУ: {len(df_ipu_all):,}")
print(f"  пустой результат ИПУ: {empty_ipu:,}")
print(f"  неклассифицированный: {uncl_ipu:,}")
print(f"Для анализа результатов ИПУ: {n_ipu:,}")

print("\n" + "=" * 70)
print(f"  РЕЗУЛЬТАТЫ ИПУ ({n_ipu:,} карточек)")
print("=" * 70)

# Сводка по классам
ipu_cls = df_ipu["ipu_class"].value_counts()
ipu_cls_pct = (ipu_cls / n_ipu * 100).round(1)
ipu_summary = pd.DataFrame({"Количество": ipu_cls, "%": ipu_cls_pct})
ipu_summary.index.name = "Класс результата ИПУ"
display(ipu_summary)

# Отдельная таблица для каждой группы
for cls_name in ["положительный", "отрицательный", "активный", "ошибка"]:
    sub = df_ipu[df_ipu["ipu_class"] == cls_name]
    if len(sub) == 0:
        continue
    detail = sub["ipu"].value_counts().reset_index()
    detail.columns = ["Результат ИПУ", "Количество"]
    detail["% от группы"] = (detail["Количество"] / len(sub) * 100).round(1)
    detail["% от всех"] = (detail["Количество"] / n_ipu * 100).round(2)
    print(f"\n{'─'*60}")
    print(f"  {cls_name.upper()} ({len(sub):,} карточек, {len(sub)/n_ipu*100:.1f}%)")
    print(f"{'─'*60}")
    display(detail.style.hide(axis="index"))

## 4. Переход от сценария к ИПУ

In [ ]:
print("=" * 70)
print("  ПЕРЕХОД ОТ СЦЕНАРИЯ К ИПУ")
print("=" * 70)

total = len(df)
to_ipu = (df["stream"] == "ipu").sum()
only_scr = total - to_ipu

print(f"\nВсего карточек:               {total:,}")
print(f"Закрыто на этапе сценария:    {only_scr:,} ({only_scr/total*100:.1f}%)")
print(f"Перешло на ИПУ:               {to_ipu:,} ({to_ipu/total*100:.1f}%)")

# Разбивка по ФП / СФП
print("\nПо типу ФП/СФП:")
trans_by_type = df.groupby("fp_type")["stream"].value_counts().unstack(fill_value=0)
trans_by_type["Всего"] = trans_by_type.sum(axis=1)
if "ipu" in trans_by_type.columns:
    trans_by_type["% на ИПУ"] = (trans_by_type["ipu"] / trans_by_type["Всего"] * 100).round(1)
display(trans_by_type)

## 5. Закрытие на стадии сценария: ФП vs СФП

In [ ]:
print("=" * 70)
print("  ЗАКРЫТИЕ НА СТАДИИ СЦЕНАРИЯ ПО ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df[df["fp_type"] == fp_t]
    n = len(sub)
    if n == 0:
        continue

    pos = (sub["scr_class"] == "положительный").sum()
    neg = (sub["scr_class"] == "отрицательный").sum()
    neu = (sub["scr_class"] == "нейтральный").sum()
    err = (sub["scr_class"] == "ошибка").sum()
    to_ipu_n = (sub["stream"] == "ipu").sum()

    print(f"\n--- {fp_t} ({n:,} карточек) ---")
    rows = [
        ["Положительный", pos, f"{pos/n*100:.1f}%"],
        ["Отрицательный", neg, f"{neg/n*100:.1f}%"],
        ["  в т.ч. перешло на ИПУ", to_ipu_n, f"{to_ipu_n/n*100:.1f}%"],
        ["Нейтральный", neu, f"{neu/n*100:.1f}%"],
        ["Ошибка", err, f"{err/n*100:.1f}%"],
    ]
    tbl = pd.DataFrame(rows, columns=["Результат сценария", "Количество", "%"])
    display(tbl.style.hide(axis="index"))

## 6. Результаты сценария по сегментам

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ СЦЕНАРИЯ ПО СЕГМЕНТАМ И ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df[df["fp_type"] == fp_t]
    if len(sub) == 0:
        continue

    print(f"\n{'='*40}")
    print(f"  {fp_t}")
    print(f"{'='*40}")

    # Абсолютные
    ct = pd.crosstab(sub["segment"], sub["scr_class"], margins=True, margins_name="Итого")
    print("\nАбсолютные значения:")
    display(ct)

    # Проценты внутри сегмента
    ct_pct = pd.crosstab(sub["segment"], sub["scr_class"], normalize="index") * 100
    ct_pct = ct_pct.round(1)
    print("\nПроценты (внутри сегмента):")
    display(ct_pct)

    # Переход на ИПУ по сегментам
    ipu_seg = sub.groupby("segment").apply(
        lambda g: pd.Series({
            "Всего": len(g),
            "На ИПУ": (g["stream"] == "ipu").sum(),
        })
    ).astype(int)
    ipu_seg["% на ИПУ"] = (ipu_seg["На ИПУ"] / ipu_seg["Всего"] * 100).round(1)
    print("\nПереход на ИПУ:")
    display(ipu_seg)

## 7. Закрытие на стадии ИПУ: ФП vs СФП

In [ ]:
print("=" * 70)
print("  ЗАКРЫТИЕ НА СТАДИИ ИПУ ПО ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df_ipu[df_ipu["fp_type"] == fp_t]
    n = len(sub)
    if n == 0:
        print(f"\n--- {fp_t}: нет карточек на этапе ИПУ ---")
        continue

    pos  = (sub["ipu_class"] == "положительный").sum()
    neg  = (sub["ipu_class"] == "отрицательный").sum()
    act  = (sub["ipu_class"] == "активный").sum()
    err  = (sub["ipu_class"] == "ошибка").sum()

    print(f"\n--- {fp_t} ({n:,} карточек на ИПУ) ---")
    rows = [
        ["Положительный", pos, f"{pos/n*100:.1f}%"],
        ["Отрицательный", neg, f"{neg/n*100:.1f}%"],
        ["Активный", act, f"{act/n*100:.1f}%"],
        ["Ошибка", err, f"{err/n*100:.1f}%"],
    ]
    tbl = pd.DataFrame(rows, columns=["Результат ИПУ", "Количество", "%"])
    display(tbl.style.hide(axis="index"))

## 8. Результаты ИПУ по сегментам

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ ИПУ ПО СЕГМЕНТАМ И ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df_ipu[df_ipu["fp_type"] == fp_t]
    if len(sub) == 0:
        print(f"\n--- {fp_t}: нет карточек на этапе ИПУ ---")
        continue

    print(f"\n{'='*40}")
    print(f"  {fp_t}")
    print(f"{'='*40}")

    ct = pd.crosstab(sub["segment"], sub["ipu_class"], margins=True, margins_name="Итого")
    print("\nАбсолютные значения:")
    display(ct)

    ct_pct = pd.crosstab(sub["segment"], sub["ipu_class"], normalize="index") * 100
    ct_pct = ct_pct.round(1)
    print("\nПроценты (внутри сегмента):")
    display(ct_pct)

## 9. Статистика по номерам ФП

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ ФП (результаты сценариев)")
print("=" * 70)

fp_only = df[df["fp_type"] == "ФП"].copy()

fp_stats = fp_only.groupby("fp_num").apply(lambda g: pd.Series({
    "Всего": len(g),
    "Положительных": (g["scr_class"] == "положительный").sum(),
    "Отрицательных": (g["scr_class"] == "отрицательный").sum(),
    "На ИПУ": (g["stream"] == "ipu").sum(),
    "Нейтральных": (g["scr_class"] == "нейтральный").sum(),
    "Ошибок": (g["scr_class"] == "ошибка").sum(),
})).astype(int).reset_index()

fp_stats["% полож."] = (fp_stats["Положительных"] / fp_stats["Всего"] * 100).round(1)
fp_stats["% отриц."] = (fp_stats["Отрицательных"] / fp_stats["Всего"] * 100).round(1)
fp_stats["% на ИПУ"] = (fp_stats["На ИПУ"] / fp_stats["Всего"] * 100).round(1)

fp_stats = fp_stats.sort_values("Всего", ascending=False)
display(
    fp_stats[
        ["fp_num", "Всего", "Положительных", "% полож.",
         "Отрицательных", "% отриц.", "На ИПУ", "% на ИПУ",
         "Нейтральных", "Ошибок"]
    ].style.hide(axis="index")
)

## 10. Статистика по номерам СФП

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ СФП (результаты сценариев)")
print("=" * 70)

sfp_only = df[df["fp_type"] == "СФП"].copy()

if len(sfp_only) == 0:
    print("Нет карточек СФП в датасете")
else:
    sfp_stats = sfp_only.groupby("fp_num").apply(lambda g: pd.Series({
        "Всего": len(g),
        "Положительных": (g["scr_class"] == "положительный").sum(),
        "Отрицательных": (g["scr_class"] == "отрицательный").sum(),
        "На ИПУ": (g["stream"] == "ipu").sum(),
        "Нейтральных": (g["scr_class"] == "нейтральный").sum(),
        "Ошибок": (g["scr_class"] == "ошибка").sum(),
    })).astype(int).reset_index()

    sfp_stats["% полож."] = (sfp_stats["Положительных"] / sfp_stats["Всего"] * 100).round(1)
    sfp_stats["% отриц."] = (sfp_stats["Отрицательных"] / sfp_stats["Всего"] * 100).round(1)
    sfp_stats["% на ИПУ"] = (sfp_stats["На ИПУ"] / sfp_stats["Всего"] * 100).round(1)

    sfp_stats = sfp_stats.sort_values("Всего", ascending=False)
    display(
        sfp_stats[
            ["fp_num", "Всего", "Положительных", "% полож.",
             "Отрицательных", "% отриц.", "На ИПУ", "% на ИПУ",
             "Нейтральных", "Ошибок"]
        ].style.hide(axis="index")
    )

## 11. Статистика по номерам ФП/СФП на стадии ИПУ

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ ФП/СФП (результаты ИПУ)")
print("=" * 70)

if len(df_ipu) == 0:
    print("Нет карточек на этапе ИПУ")
else:
    for fp_t in ["ФП", "СФП"]:
        sub = df_ipu[df_ipu["fp_type"] == fp_t]
        if len(sub) == 0:
            print(f"\n--- {fp_t}: нет карточек на ИПУ ---")
            continue

        print(f"\n{'='*40}")
        print(f"  {fp_t}")
        print(f"{'='*40}")

        ipu_stats = sub.groupby("fp_num").apply(lambda g: pd.Series({
            "Всего": len(g),
            "Положительных": (g["ipu_class"] == "положительный").sum(),
            "Отрицательных": (g["ipu_class"] == "отрицательный").sum(),
            "Активных": (g["ipu_class"] == "активный").sum(),
            "Ошибок": (g["ipu_class"] == "ошибка").sum(),
        })).astype(int).reset_index()

        ipu_stats["% полож."] = (ipu_stats["Положительных"] / ipu_stats["Всего"] * 100).round(1)
        ipu_stats["% отриц."] = (ipu_stats["Отрицательных"] / ipu_stats["Всего"] * 100).round(1)
        ipu_stats["% актив."] = (ipu_stats["Активных"] / ipu_stats["Всего"] * 100).round(1)

        ipu_stats = ipu_stats.sort_values("Всего", ascending=False)
        display(
            ipu_stats[
                ["fp_num", "Всего", "Положительных", "% полож.",
                 "Отрицательных", "% отриц.",
                 "Активных", "% актив.",
                 "Ошибок"]
            ].style.hide(axis="index")
        )

## 12. Примеры записей ИПУ: «ошибка» и «активные»

Выборка записей из дедуплицированного датасета для каждого результата из групп «ошибка» и «активный».

In [ ]:
SHOW_COLS = ["inn", "fp_num", "fp_type", "segment", "IDENTIFICATION_DATE",
             "scenario", "ipu",
             "END_DATE_SCR_FCT", "END_DATE_SCR_PLAN",
             "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT", "NEW_PLAN_END_DATE_EVT",
             "DATE_END_FP_SFP"]
show_cols = [c for c in SHOW_COLS if c in df_ipu.columns]

for cls_name, cls_label in [("ошибка", "ОШИБКА"), ("активный", "АКТИВНЫЙ")]:
    sub = df_ipu[df_ipu["ipu_class"] == cls_name]
    if len(sub) == 0:
        print(f"\n--- {cls_label}: нет карточек ---")
        continue

    for result_val in sub["ipu"].unique():
        sample = sub[sub["ipu"] == result_val]
        n_show = min(5, len(sample))
        print(f"\n{'─'*70}")
        print(f"  {cls_label}: «{result_val}»  ({len(sample):,} карточек, показано {n_show})")
        print(f"{'─'*70}")
        display(sample[show_cols].head(n_show).style.hide(axis="index"))

## 13. Техническое закрытие ФП с открытием нового сценария

Как часто уникальные карточки ФП/СФП (по `DEDUP_KEY`) имели хотя бы одну запись в `raw` с результатом сценария «Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария» (и сегментные варианты).

In [ ]:
TECH_CLOSE_SCENARIOS = [norm_text(v) for v in [
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
]]

raw_with_seg = raw.merge(
    df[DEDUP_KEY + ["segment"]].drop_duplicates(DEDUP_KEY),
    on=DEDUP_KEY, how="inner", suffixes=("", "_df")
)

has_tech = raw_with_seg[raw_with_seg["scenario"].isin(TECH_CLOSE_SCENARIOS)]
cards_with_tech = has_tech.drop_duplicates(DEDUP_KEY)

n_df = len(df)
n_tech = len(cards_with_tech)
print(f"Карточек с тех. закрытием (хотя бы 1 запись): {n_tech:,} из {n_df:,} ({n_tech/n_df*100:.1f}%)")

print("\nПо результату:")
for v in TECH_CLOSE_SCENARIOS:
    hits = has_tech[has_tech["scenario"] == v].drop_duplicates(DEDUP_KEY)
    print(f"  {len(hits):>5,}  {v}")

print("\nПо сегментам и типу ФП/СФП:")
tech_seg = cards_with_tech.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
tech_seg["Всего"] = tech_seg.sum(axis=1)
display(tech_seg)

total_seg = df.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
pct_seg = (tech_seg / total_seg * 100).round(1)
print("\n% карточек с тех. закрытием (от общего числа в сегменте):")
display(pct_seg)

## 14. ИПУ: «Требует корректировки ИПУ»

Как часто уникальные карточки ФП/СФП имели хотя бы одну запись в `raw` с результатом ИПУ:  
*«ФП не устранен в рамках Плана, оказывает негативное влияние... (требует корректировки ИПУ)»*

In [ ]:
IPU_KORR = norm_text(
    "ФП не устранен в рамках Плана, оказывает негативное влияние на исполнение "
    "участником кредитной сделки обязательств перед Банком (требует корректировки ИПУ)"
)

has_korr = raw_with_seg[raw_with_seg["ipu"] == IPU_KORR]
cards_korr = has_korr.drop_duplicates(DEDUP_KEY)

n_ipu_all = (df["stream"] == "ipu").sum()
n_korr = len(cards_korr)
print(f"Карточек с «требует корректировки ИПУ» (хотя бы 1 запись): "
      f"{n_korr:,} из {n_ipu_all:,} карточек на ИПУ ({n_korr/max(n_ipu_all,1)*100:.1f}%)")

print("\nПо сегментам и типу ФП/СФП:")
korr_seg = cards_korr.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
korr_seg["Всего"] = korr_seg.sum(axis=1)
display(korr_seg)

ipu_total_seg = df[df["stream"] == "ipu"].groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
pct_korr = (korr_seg / ipu_total_seg * 100).round(1)
print("\n% от карточек на ИПУ в сегменте:")
display(pct_korr)

## 15. Расхождение план/факт дат сценариев и ИПУ

- **Сценарий:** `END_DATE_SCR_PLAN` vs `END_DATE_SCR_FCT` (дней)
- **ИПУ:** `FIRST_END_DATE_EVENT` vs `END_EVENT_DATE_FACT` (дней)

Положительное значение = факт позже плана (просрочка), отрицательное = факт раньше плана.

In [ ]:
print("=" * 70)
print("  РАСХОЖДЕНИЕ ПЛАН / ФАКТ (дней)")
print("=" * 70)

# Сценарий: факт - план
df["_scr_plan"] = df["_END_DATE_SCR_PLAN_dt"]
df["_scr_fact"] = df["_END_DATE_SCR_FCT_dt"]
df["scr_diff_days"] = (df["_scr_fact"] - df["_scr_plan"]).dt.days

scr_diff = df.dropna(subset=["scr_diff_days"])
print(f"\nСценарий: карточек с обеими датами: {len(scr_diff):,}")

scr_agg = scr_diff.groupby(["segment", "fp_type"])["scr_diff_days"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(1)
scr_agg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
print("\nРасхождение план/факт сценария по сегментам:")
display(scr_agg)

# Общая статистика по номерам ФП
scr_by_fp = scr_diff.groupby("fp_num")["scr_diff_days"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1).reset_index()
scr_by_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
scr_by_fp = scr_by_fp.sort_values("Кол-во", ascending=False)
print("\nПо номерам ФП/СФП:")
display(scr_by_fp.style.hide(axis="index"))

# ИПУ: факт - план
df["_ipu_plan"] = df["_FIRST_END_DATE_EVENT_dt"]
df["_ipu_fact"] = df["_END_EVENT_DATE_FACT_dt"]
df["ipu_diff_days"] = (df["_ipu_fact"] - df["_ipu_plan"]).dt.days

ipu_diff = df[df["stream"] == "ipu"].dropna(subset=["ipu_diff_days"])
print(f"\n\nИПУ: карточек с обеими датами: {len(ipu_diff):,}")

if len(ipu_diff) > 0:
    ipu_agg = ipu_diff.groupby(["segment", "fp_type"])["ipu_diff_days"].agg(
        ["count", "mean", "median", "min", "max", "std"]
    ).round(1)
    ipu_agg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
    print("\nРасхождение план/факт ИПУ по сегментам:")
    display(ipu_agg)

    ipu_by_fp = ipu_diff.groupby("fp_num")["ipu_diff_days"].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(1).reset_index()
    ipu_by_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
    ipu_by_fp = ipu_by_fp.sort_values("Кол-во", ascending=False)
    print("\nПо номерам ФП/СФП:")
    display(ipu_by_fp.style.hide(axis="index"))
else:
    print("Недостаточно данных для анализа расхождения ИПУ")

## 16. Время от выявления ФП до закрытия (только сценарий)

Для карточек, которые **не переходят на ИПУ** (поток 1): время от `IDENTIFICATION_DATE` до `DATE_END_FP_SFP` (дней).

In [ ]:
print("=" * 70)
print("  ВРЕМЯ ОТ ВЫЯВЛЕНИЯ ДО ЗАКРЫТИЯ: ТОЛЬКО СЦЕНАРИЙ (поток 1)")
print("=" * 70)

scr_only = df[df["stream"] == "scenario"].copy()
scr_only["_close_dt"] = scr_only["_DATE_END_FP_SFP_dt"]
scr_only["days_to_close"] = (scr_only["_close_dt"] - scr_only["dt"]).dt.days

scr_close = scr_only.dropna(subset=["days_to_close"])
scr_close = scr_close[scr_close["days_to_close"] >= 0]
print(f"\nКарточек с обеими датами (>= 0 дней): {len(scr_close):,} из {len(scr_only):,}")

# По сегментам
agg_seg = scr_close.groupby(["segment", "fp_type"])["days_to_close"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(1)
agg_seg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
print("\nПо сегментам:")
display(agg_seg)

# По номерам ФП/СФП
agg_fp = scr_close.groupby("fp_num")["days_to_close"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1).reset_index()
agg_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
agg_fp = agg_fp.sort_values("Кол-во", ascending=False)
print("\nПо номерам ФП/СФП:")
display(agg_fp.style.hide(axis="index"))

## 17. Время от выявления ФП до закрытия (с переходом на ИПУ)

Для карточек, которые **переходят на ИПУ** (поток 2): время от `IDENTIFICATION_DATE` до `DATE_END_FP_SFP` (дней).

In [ ]:
print("=" * 70)
print("  ВРЕМЯ ОТ ВЫЯВЛЕНИЯ ДО ЗАКРЫТИЯ: С ПЕРЕХОДОМ НА ИПУ (поток 2)")
print("=" * 70)

ipu_stream = df[df["stream"] == "ipu"].copy()
ipu_stream["_close_dt"] = ipu_stream["_DATE_END_FP_SFP_dt"]
ipu_stream["days_to_close"] = (ipu_stream["_close_dt"] - ipu_stream["dt"]).dt.days

ipu_close = ipu_stream.dropna(subset=["days_to_close"])
ipu_close = ipu_close[ipu_close["days_to_close"] >= 0]
print(f"\nКарточек с обеими датами (>= 0 дней): {len(ipu_close):,} из {len(ipu_stream):,}")

# По сегментам
agg_seg = ipu_close.groupby(["segment", "fp_type"])["days_to_close"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(1)
agg_seg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
print("\nПо сегментам:")
display(agg_seg)

# По номерам ФП/СФП
agg_fp = ipu_close.groupby("fp_num")["days_to_close"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1).reset_index()
agg_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
agg_fp = agg_fp.sort_values("Кол-во", ascending=False)
print("\nПо номерам ФП/СФП:")
display(agg_fp.style.hide(axis="index"))

# Сравнение двух потоков
print("\n" + "=" * 70)
print("  СРАВНЕНИЕ: ТОЛЬКО СЦЕНАРИЙ vs С ИПУ")
print("=" * 70)
comp = pd.DataFrame({
    "Только сценарий": scr_close["days_to_close"].describe(),
    "С переходом на ИПУ": ipu_close["days_to_close"].describe(),
}).round(1)
display(comp)